# Deep Learning Predict Market Direction 

In [46]:
# Data 
import yfinance as yf 

# Library 
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt 
import seaborn as sns
import plotly.express as px 

In [47]:
# USD/EUR price levels for the past 10 years 
symbol = "BTC-USD"

btc = yf.download(symbol, start="2016-01-01", end="2026-01-01", interval='1d')["Close"]

btc

[*********************100%***********************]  1 of 1 completed


Ticker,BTC-USD
Date,
2016-01-01,434.334015
2016-01-02,433.437988
2016-01-03,430.010986
2016-01-04,433.091003
2016-01-05,431.959991
...,...
2025-12-27,87802.156250
2025-12-28,87835.835938
2025-12-29,87138.140625


In [48]:
btc = pd.DataFrame(btc)
btc.columns

Index(['BTC-USD'], dtype='object', name='Ticker')

In [49]:
btc.index = pd.to_datetime(btc.index)

In [50]:
btc['BTC-USD'].shift(1)

Date
2016-01-01             NaN
2016-01-02      434.334015
2016-01-03      433.437988
2016-01-04      430.010986
2016-01-05      433.091003
                  ...     
2025-12-27    87301.429688
2025-12-28    87802.156250
2025-12-29    87835.835938
2025-12-30    87138.140625
2025-12-31    88430.132812
Name: BTC-USD, Length: 3653, dtype: float64

In [51]:
# Calculate the returns and add it to dataframe 
btc['return'] = np.log(btc['BTC-USD']/btc['BTC-USD'].shift(1))

# When the market direction is greater than 0 => classify 1, less than 0 --> 0
btc['direction'] = np.where(btc['return'] > 0, 1, 0)

In [52]:
btc

Ticker,BTC-USD,return,direction
Date,,,
2016-01-01,434.334015,NaN,0
2016-01-02,433.437988,-0.002065,0
2016-01-03,430.010986,-0.007938,0
2016-01-04,433.091003,0.007137,1
2016-01-05,431.959991,-0.002615,0
...,...,...,...
2025-12-27,87802.156250,0.005719,1
2025-12-28,87835.835938,0.000384,1
2025-12-29,87138.140625,-0.007975,0


In [53]:
# Create 5 columns for each lag representing previous day's return 
lags = 5

cols = []
for lag in range(1, lags + 1):
    col_name = f"lag_{lag}"
    btc[col_name] = btc['return'].shift(lag)
    cols.append(col_name)

btc.dropna(inplace=True)

In [54]:
btc

Ticker,BTC-USD,return,direction,lag_1,lag_2,lag_3,lag_4,lag_5
Date,,,,,,,,
2016-01-07,458.048004,0.065272,1,-0.006631,-0.002615,0.007137,-0.007938,-0.002065
2016-01-08,453.230011,-0.010574,0,0.065272,-0.006631,-0.002615,0.007137,-0.007938
2016-01-09,447.610992,-0.012475,0,-0.010574,0.065272,-0.006631,-0.002615,0.007137
2016-01-10,447.990997,0.000849,1,-0.012475,-0.010574,0.065272,-0.006631,-0.002615
2016-01-11,448.428009,0.000975,1,0.000849,-0.012475,-0.010574,0.065272,-0.006631
...,...,...,...,...,...,...,...,...
2025-12-27,87802.156250,0.005719,1,0.000764,-0.004315,0.002262,-0.012234,-0.001488
2025-12-28,87835.835938,0.000384,1,0.005719,0.000764,-0.004315,0.002262,-0.012234
2025-12-29,87138.140625,-0.007975,0,0.000384,0.005719,0.000764,-0.004315,0.002262


# Deep Neural Network


In [60]:
import tensorflow as tf 
from keras.models import Sequential
from keras.layers import Dense, Dropout
from keras.optimizers import Adam, RMSprop 
import random

In [56]:
optimizer = Adam(learning_rate=0.0001)

def set_seeds(seed=100):
    random.seed(100)
    np.random.seed(seed)
    tf.random.set_seed(100)

2026-01-24 15:13:04.642449: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [ ]:
set_seeds()
model = Sequential()
model.add(Dense(64, activation='relu', input_shape=(lags, )))
model.add(Dropout(0.5))
model.add(Dense(64, activation='relu'))
model.add(Dropout(0.5))
model.add(Dense(1, activation='sigmoid'))

model.compile(optimizer=optimizer,
              loss='binary_crossentropy',
              metrics=['accuracy'])



# model = keras.Sequential([
#     keras.layers.Dense(60, input_dim = 60, activation = 'relu'),
#     keras.layers.Dropout(0.5),
#     keras.layers.Dense(30, activation = 'relu'),
#     keras.layers.Dropout(0.5),
#     keras.layers.Dense(15, activation = 'relu'),
#     keras.layers.Dropout(0.5),
#     keras.layers.Dense(1, activation = 'sigmoid')
# ])
# model.compile(
#     loss = 'binary_crossentropy',
#     optimizer = 'adam',
#     metrics = ['accuracy']

# )

/home/xuanhoang/anaconda3/envs/my_env/lib/python3.12/site-packages/keras/src/layers/core/dense.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


NameError: name 'Dropout' is not defined